# Pairwise Global Alignment

Goal is to create a set of pairwise global alignments based on the mixed sampels with the correct references.

-   Align all sequences to the 2 possible references (global alignment with muscle)
-   Use Biopython `alignment.counts()` to summarise the following:
    -   Number of identities
    -   Number of mismatches
    -   Number of gaps


In [20]:
from Bio.Phylo.TreeConstruction import DistanceCalculator
from Bio import SeqIO, AlignIO
import pandas as pd
import os
import sys
import subprocess
import re

In [21]:
!pip show biopython

Name: biopython
Version: 1.85
Summary: Freely available tools for computational molecular biology.
Home-page: https://biopython.org/
Author: The Biopython Contributors
Author-email: biopython@biopython.org
License: 
Location: /Users/joonklaps/opt/anaconda3/envs/intrahost-env/lib/python3.13/site-packages
Requires: numpy
Required-by: AlignmentViewer, dash_bio


In [24]:
got_dict = {
    "Eddard": ["MN090188.1", "MN090277.1"],
    "Catelyn": ["MN090188.1", "MN090277.1"],
    "Robb": ["MN090188.1", "MN090277.1"],
    "Jon": ["MN090240.1", "MN090277.1"],
    "Sansa": ["MN090240.1", "MN090277.1"],
    "Arya": ["MN090240.1", "MN090277.1"],
    "Daenerys": ["MZ766668.1", "MN090277.1"],
    "Tyrion": ["MZ766668.1", "MN090277.1"],
    "Jaime": ["MZ766668.1", "MN090277.1"],
    "Bran": ["MN090277.1"],
    "Rickon": ["MN090188.1"],
    "Theon": ["MN090240.1"],
    "Jorah": ["MZ766668.1"],
}

skiphybrid_ri = SeqIO.to_dict(SeqIO.parse("./data/hybrid-vs-skiphybrid/skiphybrid.unique.fasta", "fasta"))
noskiphybrid_ri = SeqIO.to_dict(SeqIO.parse("./data/hybrid-vs-skiphybrid/skiphybrid.unique.fasta", "fasta"))
consensus_cdhit_ri = SeqIO.to_dict(SeqIO.parse("./data/influence-reference-cdhit/unique.fasta", "fasta"))
consensus_mmseqs_ri = SeqIO.to_dict(SeqIO.parse("./data/influence-reference-mmseqs/combined-consensus.fasta", "fasta"))
reference_ri = SeqIO.to_dict(SeqIO.parse("./data/ref.unique.fasta", "fasta"))

In [18]:
def align_with_mafft(input, output):
    mafft_path = "/Users/joonklaps/opt/anaconda3/bin/mafft"
    # mafft_path = "/opt/conda/bin/mafft"  # `which mafft` should return this path
    cline = f"{mafft_path} --globalpair --maxiterate 1000 --adjustdirection --thread -1 {input} > {output}"
    prog = subprocess.run(cline, shell=True, check=True, stderr=subprocess.PIPE)

    if os.path.isfile(output):
        print(f"Alignment successful: {output}")
    else:
        print("Alignment failed, output file does not exist.")
        print(f"Command: {cline}")
        print(f"Error: {prog.stderr.decode()}")
        sys.exit(1)


def generate_alignment(dir, record, reference):
    # Create alignments directory if it doesn't exist
    os.makedirs(dir, exist_ok=True)

    fasta = dir + f"/{record.id}.vs.{reference.id}.fasta".replace("-", ".").replace("|", "_")
    alignment = dir + f"/{record.id}.vs.{reference.id}.aln".replace("-", ".").replace("|", "_")

    if os.path.isfile(alignment):
        with open(alignment) as f:
            content = f.read()
            if content.count(">") == 2:
                return alignment

    if not os.path.isfile(fasta):
        with open(fasta, "w") as f:
            f.write(f">{record.id}\n{record.seq}\n>{reference.id}\n{reference.seq}\n")

    align_with_mafft(fasta, alignment)
    return alignment

In [6]:
def compute_snp_mismatch_stats(seq_a: str, seq_b: str):
    """
    Compute SNP-based mismatch statistics between two aligned sequences.

    Rules:
    - Only count positions where both bases are valid nucleotides (A/C/G/T)
    - Ignore positions where either base is 'N' (any case) or a gap ('-' or '.')
    - Mismatch = valid base vs different valid base (i.e., introduced SNP)

    Returns a dict with:
    - snp_mismatches: number of SNP mismatches
    - snp_matches: number of matches among comparable sites
    - snp_comparable_sites: number of sites compared (A/C/G/T vs A/C/G/T)
    - snp_mismatch_rate: mismatches / comparable_sites (None if no comparable sites)
    """
    valid = {"A", "C", "G", "T"}
    mismatches = 0
    matches = 0
    comparable = 0

    # Ensure equal length; if not, compare up to shortest just in case
    for a, b in zip(str(seq_a).upper(), str(seq_b).upper()):
        if a in valid and b in valid:
            comparable += 1
            if a == b:
                matches += 1
            else:
                mismatches += 1
        else:
            # skip positions with N/gaps/others
            continue

    rate = (mismatches / comparable) if comparable else None
    return {
        "snp_mismatches": mismatches,
        "snp_matches": matches,
        "snp_comparable_sites": comparable,
        "snp_mismatch_rate": rate,
    }

In [28]:
def generate_pairwise_alignments(target_records, reference_records, got_mapping, output_dir):
    """
    Generate pairwise alignments between target records and their corresponding reference sequences.

    Parameters:
    -----------
    target_records : dict
        Dictionary of target sequences to align (SeqRecord objects)
    reference_records : dict
        Dictionary of reference sequences (SeqRecord objects)
    got_mapping : dict
        Dictionary mapping GOT characters to reference IDs
    output_dir : str
        Directory to store the alignment files

    Returns:
    --------
    dict
        Dictionary of alignment statistics for each target-reference pair
    """
    result_dict = {}
    for _, record in target_records.items():
        parts = record.id.split("-")
        if len(parts) >= 2:
            got = re.match(r"([A-Za-z]+)", parts[0]).group(1)
            try:
                other = "-".join(parts[1:])
                if not "it2" in other:
                    continue

                db = re.match(r"((M[NZ][0-9\.]+(\-M[NZ][0-9\.]+)?)|virosaurus|reference|URVDBv29.0)", other).group(1)
            except AttributeError as e:
                print(f"Skipping {record.id} due to invalid ID format")
                raise ValueError(f"Invalid ID format for {record.id}", e)

            # Skip if got is not in mapping
            if got not in got_mapping:
                print(f"No reference mapping found for {got}")
                continue

            for refid in got_mapping[got]:
                print(f"{record.id} vs {refid} ...")
                reference = reference_records.get(refid)

                # Skip if reference not found
                if reference is None:
                    print(f"Reference {refid} not found")
                    continue

                alignment_path = generate_alignment(output_dir, record, reference)

                if alignment_path is None:
                    print(f"      Skipping {record.id} vs {refid} due to alignment failure")
                    continue

                try:
                    pwa = AlignIO.read(alignment_path, "fasta", 2)

                    # Compute SNP-based mismatch stats on the two aligned sequences
                    snp_stats = compute_snp_mismatch_stats(pwa[0].seq, pwa[1].seq)

                    result_dict[(record.id, refid)] = {
                        "mismatches": snp_stats["snp_mismatches"],  # override with SNP-based mismatches
                        "identities": snp_stats["snp_matches"],
                        "query_length": len(pwa[0].seq),  # Length of the first sequence
                        "number_of_gaps": pwa[0].count("-"),  # Number of gaps in the first sequence
                        "number_of_Ns": pwa[0].count("n") + pwa[0].count("N"),  # Number of Ns in the first sequence
                        "alignment_length": pwa.alignment.length,
                        "got": got,
                        "db": db,
                        "alignment_path": alignment_path,
                    }
                except Exception as e:
                    print(f"Error processing alignment for {record.id} vs {refid}")
                    print(f"Error: {str(e)}")
        else:
            print(f"Skipping {record.id} due to invalid ID format")

    return result_dict

In [8]:
skip_hybrid_dict = generate_pairwise_alignments(
    target_records=skiphybrid_ri,
    reference_records=reference_ri,
    got_mapping=got_dict,
    output_dir="./data/hybrid-vs-skiphybrid/skiphybrid-vs-ref/",
)

Arya-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090277.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-virosaurus_cl1_it2.consensus_ivar vs MN090240.1 ...
Arya-virosaurus_cl1_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090240_cl0

In [9]:
noskip_hybrid_dict = generate_pairwise_alignments(
    target_records=noskiphybrid_ri,
    reference_records=reference_ri,
    got_mapping=got_dict,
    output_dir="./data/hybrid-vs-skiphybrid/hybrid-vs-ref/",
)

Arya-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090277.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-virosaurus_cl1_it2.consensus_ivar vs MN090240.1 ...
Arya-virosaurus_cl1_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090240_cl0

In [31]:
influence_ref_dict_cdhit = generate_pairwise_alignments(
    target_records=consensus_cdhit_ri,
    reference_records=reference_ri,
    got_mapping=got_dict,
    output_dir="./data/influence-reference-cdhit/alignment-vs-ref/",
)

Arya-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-MN090277_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl0_it2.consensus_ivar vs MN090277.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090240.1 ...
Arya-reference_cl1_it2.consensus_ivar vs MN090277.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090240.1 ...
Arya-virosaurus_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090188_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MN090240_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN090277-MZ766668_cl0_it2.consensus_ivar vs MN090277.1 ...
Bran-MN09027

In [29]:
influence_ref_dict_mmseqs = generate_pairwise_alignments(
    target_records=consensus_mmseqs_ri,
    reference_records=reference_ri,
    got_mapping=got_dict,
    output_dir="./data/influence-reference-mmseqs/alignment-vs-ref/",
)

Arya-MN090188.1_cl0_it2.consensus_bcftools vs MN090240.1 ...
Arya-MN090188.1_cl0_it2.consensus_bcftools vs MN090277.1 ...
Bran-MN090188.1_cl0_it2.consensus_bcftools vs MN090277.1 ...
Catelyn-MN090188.1_cl10_it2.consensus_bcftools vs MN090188.1 ...
Catelyn-MN090188.1_cl10_it2.consensus_bcftools vs MN090277.1 ...
Daenerys-MN090188.1_cl2_it2.consensus_bcftools vs MZ766668.1 ...
Daenerys-MN090188.1_cl2_it2.consensus_bcftools vs MN090277.1 ...
Daenerys-MN090188.1_cl3_it2.consensus_bcftools vs MZ766668.1 ...
Daenerys-MN090188.1_cl3_it2.consensus_bcftools vs MN090277.1 ...
Eddard-MN090188.1_cl42_it2.consensus_bcftools vs MN090188.1 ...
Eddard-MN090188.1_cl42_it2.consensus_bcftools vs MN090277.1 ...
Jaime-MN090188.1_cl0_it2.consensus_bcftools vs MZ766668.1 ...
Jaime-MN090188.1_cl0_it2.consensus_bcftools vs MN090277.1 ...
Jaime-MN090188.1_cl50_it2.consensus_bcftools vs MZ766668.1 ...
Jaime-MN090188.1_cl50_it2.consensus_bcftools vs MN090277.1 ...
Jon-MN090188.1_cl57_it2.consensus_bcftools vs MN0

In [32]:
pd.DataFrame.from_dict(noskip_hybrid_dict, orient="index").to_csv("./data/hybrid-vs-skiphybrid/hybrid-vs-ref.csv")
pd.DataFrame.from_dict(skip_hybrid_dict, orient="index").to_csv("./data/hybrid-vs-skiphybrid/skiphybrid-vs-ref.csv")
pd.DataFrame.from_dict(influence_ref_dict_cdhit, orient="index").to_csv("./data/influence-reference-cdhit/consensus-vs-ref.csv")
pd.DataFrame.from_dict(influence_ref_dict_mmseqs, orient="index").to_csv("./data/influence-reference-mmseqs/consensus-vs-ref.csv")